# **Data Science 112 Final Project: Data Extraction**



# Data Collection

This project uses Film Corpus 2.0: https://nlds.soe.ucsc.edu/fc2

We are taking the full scripts of each movie from the corpus and splitting them up by scene.

In [ ]:
import re
from pathlib import Path
import pandas as pd

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
PROJECT_DIR = Path("/content/drive/MyDrive/DATASCI 112: Final Project - Movie Emotions by Genre")
DATA_DIR = PROJECT_DIR / "data"

FULL_SCRIPT_DIR = DATA_DIR / "full_scripts"
SCENE_ONLY_DIR = DATA_DIR / "scene_only"

OUTPUT_DIR = PROJECT_DIR / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

SCENE_OUTPUT_PATH = OUTPUT_DIR / "scene_level_dialogue_simplified.csv"
LOG_OUTPUT_PATH = OUTPUT_DIR / "scene_extraction_log.csv"

print("Full script files:", len(list(FULL_SCRIPT_DIR.glob("*.txt"))))
print("Scene-only files:", len(list(SCENE_ONLY_DIR.glob("*.txt"))))

Full script files: 1070
Scene-only files: 960


In [ ]:
def read_text(path):
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        return f.read()

def clean_text(text):
    text = re.sub(r"<[^>]+>", "", text)
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    return text.strip()

def normalize_line(line):
    line = line.upper()
    line = re.sub(r"\s+", " ", line)
    return line.strip()

In [ ]:
scene_heading_pattern = r"^\s*(INT\.|EXT\.|INT/EXT\.|EXT/INT\.)\s+.+"

def get_scene_headings(scene_text):
    headings = []
    for line in scene_text.splitlines():
        line = line.strip()
        if re.match(scene_heading_pattern, line, flags=re.IGNORECASE):
            headings.append(line)
    return headings

In [ ]:
def split_into_scenes(full_text, scene_headings):
    full_lines = full_text.splitlines()
    heading_set = set(normalize_line(h) for h in scene_headings)
    scene_positions = []
    char_position = 0
    for line in full_lines:
        if normalize_line(line.strip()) in heading_set:
            scene_positions.append({
                "scene_heading": line.strip(),
                "start_char": char_position
            })
        char_position += len(line) + 1
    scenes = []

    for i in range(len(scene_positions)):
        start = scene_positions[i]["start_char"]
        if i + 1 < len(scene_positions):
            end = scene_positions[i + 1]["start_char"]
        else:
            end = len(full_text)
        scenes.append({
            "scene_number": i + 1,
            "scene_heading": scene_positions[i]["scene_heading"],
            "scene_text": full_text[start:end]
        })

    return scenes

In [ ]:
def looks_like_character_name(line):
    line = line.strip()
    if line == "":
        return False
    if re.match(scene_heading_pattern, line, flags=re.IGNORECASE):
        return False
    if len(line.split()) > 6:
        return False
    if line.startswith("(") and line.endswith(")"):
        return False
    return line.isupper()


def extract_dialogue(scene_text):
    lines = scene_text.splitlines()
    dialogue_parts = []
    current_speaker = None
    current_dialogue = []
    for line in lines:
        line = line.strip()
        if line == "":
            continue
        if looks_like_character_name(line):
            if current_speaker is not None and len(current_dialogue) > 0:
                dialogue_parts.append(" ".join(current_dialogue))
            current_speaker = line
            current_dialogue = []
        elif current_speaker is not None:
            if not (line.startswith("(") and line.endswith(")")):
                current_dialogue.append(line)
    if current_speaker is not None and len(current_dialogue) > 0:
        dialogue_parts.append(" ".join(current_dialogue))
    return " ".join(dialogue_parts)

In [ ]:
all_rows = []
log_rows = []

full_script_files = sorted(FULL_SCRIPT_DIR.glob("*.txt"))

for full_path in full_script_files:
    movie_id = full_path.stem
    scene_path = SCENE_ONLY_DIR / f"{movie_id}_scene.txt"

    print("Processing:", movie_id)
    if not scene_path.exists():
        log_rows.append({
            "movie_id": movie_id,
            "status": "skipped",
            "reason": "missing scene-only file"
        })
        continue
    full_text = clean_text(read_text(full_path))
    scene_text = clean_text(read_text(scene_path))
    scene_headings = get_scene_headings(scene_text)
    if len(scene_headings) == 0:
        log_rows.append({
            "movie_id": movie_id,
            "status": "skipped",
            "reason": "no scene headings found"
        })
        continue
    scenes = split_into_scenes(full_text, scene_headings)
    if len(scenes) == 0:
        log_rows.append({
            "movie_id": movie_id,
            "status": "skipped",
            "reason": "could not split full script into scenes"
        })
        continue
    for scene in scenes:
        dialogue_text = extract_dialogue(scene["scene_text"])
        all_rows.append({
            "movie_id": movie_id,
            "scene_number": scene["scene_number"],
            "scene_heading": scene["scene_heading"],
            "relative_position": scene["scene_number"] / len(scenes),
            "dialogue_text": dialogue_text
        })
    log_rows.append({
        "movie_id": movie_id,
        "status": "processed",
        "reason": None,
        "num_scenes": len(scenes)
    })

scene_df = pd.DataFrame(all_rows)

log_df = pd.DataFrame(log_rows)

scene_df.to_csv(SCENE_OUTPUT_PATH, index=False)
log_df.to_csv(LOG_OUTPUT_PATH, index=False)

scene_df.head()

Processing: 10thingsihateaboutyou
Processing: 12
Processing: 127hours
Processing: 12andholding
Processing: 12monkeys
Processing: 12yearsaslave
Processing: 1492conquestofparadise
Processing: 15minutes
Processing: 17again
Processing: 187
Processing: 2001aspaceodyssey
Processing: 2012
Processing: 30minutesorless
Processing: 42
Processing: 44inchchest
Processing: 48hrs.
Processing: 500daysofsummer
Processing: 5050
Processing: 8mm
Processing: 9
Processing: abovethelaw
Processing: absolutepower
Processing: abyssthe
Processing: aceventurapetdetective
Processing: adaptation
Processing: addamsfamilythe
Processing: adjustmentbureauthe
Processing: adventuresofbuckaroobanzaiacrosstheeighthdimensionthe
Processing: afewgoodmen
Processing: affliction
Processing: after.life
Processing: afterschoolspecial
Processing: agnesofgod
Processing: airforceone
Processing: airplane
Processing: airplane2thesequel
Processing: aladdin
Processing: ali
Processing: alien
Processing: alien3
Processing: aliennation
Proc

,movie_id,scene_number,scene_heading,relative_position,dialogue_text
0,10thingsihateaboutyou,1,INT. GIRLS' ROOM - DAY,0.011236,Did you change your hair? No. You might wanna ...
1,10thingsihateaboutyou,2,INT. HALLWAY - DAY- CONTINUOUS,0.022472,We've got your basic beautiful people. Unless ...
2,10thingsihateaboutyou,3,EXT. SCHOOL COURTYARD - DAY,0.033708,And these delusionals are the White Rastae. Se...
3,10thingsihateaboutyou,4,INT. CAFETERIA - DAY - CONTINUOUS,0.044944,"Future MBAs- We're all Ivy League, already ac..."
4,10thingsihateaboutyou,5,INT. GUIDANCE COUNSELOR'S OFFICE - DAY,0.056180,"Katarina Stratford. My, my. You've been terr..."


# Data Merging

This project uses the TMDB API: https://developer.themoviedb.org/docs/getting-started

We are taking the scene-by-scene dataset and merging it with information about genre and runtime from the TMDB API.

Summary:
1. Create better search queries from movie_id.
2. Search TMDb.
3. Collect multiple candidate results.
4. Pick the candidate whose title most closely matches movie_id.
5. Reject weak matches instead of forcing bad ones.


In [ ]:
!pip -q install rapidfuzz wordninja

import wordninja
from rapidfuzz import fuzz

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 541.6/541.6 kB 2.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 16.7 MB/s eta 0:00:00


In [ ]:
API_METADATA_PATH = OUTPUT_DIR / "tmdb_api_movie_metadata.csv"
API_UNMATCHED_PATH = OUTPUT_DIR / "tmdb_api_unmatched_movies.csv"
MERGED_OUTPUT_PATH = OUTPUT_DIR / "scene_level_dialogue_with_metadata.csv"

domain = "https://api.themoviedb.org/3"
domain = "https://api.themoviedb.org/3"

TMDB_BEARER_TOKEN = "PASTE TOKEN HERE"

headers = {
    "accept": "application/json",
    "Authorization": f"Bearer {"PASTE TOKEN HERE"}"
}

In [ ]:
def normalize_special_numbers(text):
    text = str(text)
    text = text.replace("²", "2").replace("³", "3").replace("¹", "1")
    return text

def compact_title(title):
    title = normalize_special_numbers(title).lower()
    title = title.replace("&", "and")
    title = re.sub(r"\b\d{4}\b", "", title)
    title = re.sub(r"[^a-z0-9]", "", title)
    return title.strip()

def basic_clean_title(title):
    title = normalize_special_numbers(title).lower()
    title = title.replace("_", " ").replace("-", " ")
    title = re.sub(r"([a-z])([0-9])", r"\1 \2", title)
    title = re.sub(r"([0-9])([a-z])", r"\1 \2", title)
    title = re.sub(r"[^a-z0-9\s]", " ", title)
    title = re.sub(r"\s+", " ", title)
    return title.strip()

def make_search_queries(movie_id):
    base = basic_clean_title(movie_id)
    split = " ".join(wordninja.split(str(movie_id).lower()))
    split = basic_clean_title(split)

    queries = [base, split]
    return list(dict.fromkeys([q for q in queries if q]))

In [ ]:
def search_tmdb(query, pages=2):
    """
    Search TMDb and collect results from the first few pages.
    More pages helps because the best match is not always first.
    """
    all_results = []

    for page in range(1, pages + 1):
        url = f"{domain}/search/movie"

        params = {
            "query": query,
            "include_adult": "false",
            "language": "en-US",
            "page": page
        }

        response = requests.get(url, headers=headers, params=params)

        if response.status_code != 200:
            print("Search error:", response.status_code, query)
            break

        data = response.json()
        results = data.get("results", [])
        all_results.extend(results)

        if page >= data.get("total_pages", 1):
            break

        time.sleep(0.05)

    return all_results


def get_movie_details(tmdb_id):
    """
    Get detailed metadata from TMDb.
    Details include runtime and genres.
    """
    url = f"{domain}/movie/{tmdb_id}"

    params = {
        "language": "en-US"
    }

    response = requests.get(url, headers=headers, params=params)

    if response.status_code != 200:
        print("Details error:", response.status_code, tmdb_id)
        return None

    return response.json()

In [ ]:
def score_candidate(movie_id, query, candidate):
    movie_key = compact_title(movie_id)
    query_key = compact_title(query)

    title_key = compact_title(candidate.get("title", ""))
    original_key = compact_title(candidate.get("original_title", ""))

    candidate_keys = [title_key, original_key]

    if movie_key in candidate_keys or query_key in candidate_keys:
        title_score = 100
    else:
        title_score = max(
            fuzz.ratio(movie_key, title_key),
            fuzz.ratio(movie_key, original_key),
            fuzz.ratio(query_key, title_key),
            fuzz.ratio(query_key, original_key)
        )

    query_words = set(basic_clean_title(query).split())
    candidate_words = set(basic_clean_title(candidate.get("title", "")).split())

    extra_word_penalty = len(candidate_words - query_words) * 5
    popularity_bonus = min(float(candidate.get("popularity", 0)) / 20, 5)

    return title_score - extra_word_penalty + popularity_bonus, title_score

def choose_best_candidate(movie_id, all_candidates):
    if len(all_candidates) == 0:
        return None

    scored = []

    for item in all_candidates:
        final_score, title_score = score_candidate(
            movie_id,
            item["query"],
            item["candidate"]
        )

        scored.append({
            "query": item["query"],
            "candidate": item["candidate"],
            "final_score": final_score,
            "title_score": title_score
        })

    best = sorted(scored, key=lambda x: x["final_score"], reverse=True)[0]

    if best["title_score"] < 75:
        return None

    return best

In [ ]:
movies_df = scene_df[["movie_id"]].drop_duplicates().copy()

metadata_rows = []
unmatched_rows = []

for i, row in movies_df.iterrows():
    movie_id = row["movie_id"]
    print(f"{i + 1}/{len(movies_df)}: {movie_id}")

    all_candidates = []

    for query in make_search_queries(movie_id):
        for result in search_tmdb(query, pages=2):
            all_candidates.append({
                "query": query,
                "candidate": result
            })

        time.sleep(0.05)

    best = choose_best_candidate(movie_id, all_candidates)

    if best is None:
        unmatched_rows.append({
            "movie_id": movie_id,
            "reason": "no strong title match"
        })
        continue

    tmdb_id = best["candidate"]["id"]
    details = get_movie_details(tmdb_id)

    if details is None:
        unmatched_rows.append({
            "movie_id": movie_id,
            "reason": "could not get details"
        })
        continue

    genres = ", ".join([g["name"] for g in details.get("genres", [])])

    metadata_rows.append({
        "movie_id": movie_id,
        "query_used": best["query"],
        "tmdb_id": tmdb_id,
        "title": details.get("title"),
        "runtime": details.get("runtime"),
        "genre": genres,
        "release_date": details.get("release_date"),
        "popularity": details.get("popularity"),
        "title_score": best["title_score"],
        "final_score": best["final_score"]
    })

    time.sleep(0.1)

tmdb_metadata_df = pd.DataFrame(metadata_rows)
tmdb_unmatched_df = pd.DataFrame(unmatched_rows)

tmdb_metadata_df.to_csv(API_METADATA_PATH, index=False)
tmdb_unmatched_df.to_csv(API_UNMATCHED_PATH, index=False)

print("Matched movies:", tmdb_metadata_df["movie_id"].nunique())
print("Unmatched movies:", tmdb_unmatched_df["movie_id"].nunique())

1/685: 10thingsihateaboutyou
90/685: 127hours
302/685: 12andholding
462/685: 12monkeys
633/685: 12yearsaslave
724/685: 1492conquestofparadise
880/685: 15minutes
945/685: 17again
1062/685: 2012
1317/685: 30minutesorless
1422/685: 500daysofsummer
1443/685: 5050
1539/685: 8mm
1821/685: abyssthe
2049/685: adaptation
2251/685: adjustmentbureauthe
2492/685: afewgoodmen
2570/685: affliction
2649/685: airforceone
2986/685: airplane
3321/685: airplane2thesequel
3578/685: ali
3772/685: alien3
3818/685: aliennation
3987/685: aliens
4176/685: allabouteve
4230/685: allaboutsteve
4423/685: aloneinthedark
4471/685: amadeus
4645/685: amelia
4730/685: americanbeauty
4892/685: americangangster
4893/685: americanhistoryx
5030/685: americanhustle
5062/685: americanpie
5272/685: americanpresidentthe
5383/685: americanshaolinkingofkickboxersii
5533/685: americansplendor
5668/685: americanwerewolfinlondon
5677/685: analyzethat
5801/685: analyzethis
5808/685: anastasia
5920/685: angeleyes
6030/685: annakareni

In [ ]:
scene_api_metadata_df = scene_df.merge(
    tmdb_metadata_df,
    on="movie_id",
    how="left"
)

scene_api_metadata_df.to_csv(MERGED_OUTPUT_PATH, index=False)

print("Saved merged dataset to:", MERGED_OUTPUT_PATH)
print("Total movies:", scene_api_metadata_df["movie_id"].nunique())
print("Movies with metadata:", scene_api_metadata_df[scene_api_metadata_df["title"].notna()]["movie_id"].nunique())
print("Movies without metadata:", scene_api_metadata_df[scene_api_metadata_df["title"].isna()]["movie_id"].nunique())

scene_api_metadata_df.head()

Saved merged dataset to: /content/drive/MyDrive/DATASCI 112: Final Project - Movie Emotions by Genre/outputs/scene_level_dialogue_with_metadata.csv
Total movies: 685
Movies with metadata: 597
Movies without metadata: 88


,movie_id,scene_number,scene_heading,relative_position,dialogue_text,query_used,tmdb_id,title,runtime,genre,release_date,popularity,title_score,final_score
0,10thingsihateaboutyou,1,INT. GIRLS' ROOM - DAY,0.011236,Did you change your hair? No. You might wanna ...,10 things i hate about you,4951.0,10 Things I Hate About You,97.0,"Comedy, Romance, Drama",1999-03-30,15.3798,100.0,100.76899
1,10thingsihateaboutyou,2,INT. HALLWAY - DAY- CONTINUOUS,0.022472,We've got your basic beautiful people. Unless ...,10 things i hate about you,4951.0,10 Things I Hate About You,97.0,"Comedy, Romance, Drama",1999-03-30,15.3798,100.0,100.76899
2,10thingsihateaboutyou,3,EXT. SCHOOL COURTYARD - DAY,0.033708,And these delusionals are the White Rastae. Se...,10 things i hate about you,4951.0,10 Things I Hate About You,97.0,"Comedy, Romance, Drama",1999-03-30,15.3798,100.0,100.76899
3,10thingsihateaboutyou,4,INT. CAFETERIA - DAY - CONTINUOUS,0.044944,"Future MBAs- We're all Ivy League, already ac...",10 things i hate about you,4951.0,10 Things I Hate About You,97.0,"Comedy, Romance, Drama",1999-03-30,15.3798,100.0,100.76899
4,10thingsihateaboutyou,5,INT. GUIDANCE COUNSELOR'S OFFICE - DAY,0.056180,"Katarina Stratford. My, my. You've been terr...",10 things i hate about you,4951.0,10 Things I Hate About You,97.0,"Comedy, Romance, Drama",1999-03-30,15.3798,100.0,100.76899
